In [1]:
!pip -q install stratify

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import shapely
import warnings

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

In [4]:
coloc_gdf = gpd.read_parquet("orcestra_coloc_ec_sounders.parquet")

In [5]:
coloc_gdf

,index_coloc,frame,idx,hdim_1,hdim_2,num,threshold_value,max_precip,feature,time,...,index_n20,results_n20,start_time_n20,end_time_n20,index_snpp,results_snpp,start_time_snpp,end_time_snpp,time_diff_n20,time_diff_snpp
0,35492,41,1289,2589.0,520.30,10,20,25.310354,35493,2024-08-10 20:30:00,...,54,{'meta': {'collection-concept-id': 'C256086262...,2024-08-10 20:36:00,2024-08-10 20:42:00,51,{'meta': {'collection-concept-id': 'C255991929...,2024-08-10 20:12:00,2024-08-10 20:18:00,-1 days +23:54:32,0 days 00:18:32
1,35675,42,155,750.0,662.00,1,5,5.148579,35676,2024-08-10 21:00:00,...,52,{'meta': {'collection-concept-id': 'C256086262...,2024-08-10 19:54:00,2024-08-10 20:00:00,49,{'meta': {'collection-concept-id': 'C255991929...,2024-08-10 19:30:00,2024-08-10 19:36:00,0 days 01:11:00,0 days 01:35:00
2,36208,42,945,745.0,638.00,1,10,11.937240,36209,2024-08-10 21:00:00,...,52,{'meta': {'collection-concept-id': 'C256086262...,2024-08-10 19:54:00,2024-08-10 20:00:00,49,{'meta': {'collection-concept-id': 'C255991929...,2024-08-10 19:30:00,2024-08-10 19:36:00,0 days 01:11:00,0 days 01:35:00
3,68260,78,704,3384.0,664.50,4,5,6.894357,68261,2024-08-11 15:00:00,...,143,{'meta': {'collection-concept-id': 'C256086262...,2024-08-11 15:06:00,2024-08-11 15:12:00,140,{'meta': {'collection-concept-id': 'C255991929...,2024-08-11 14:42:00,2024-08-11 14:48:00,0 days 00:07:22,0 days 00:31:22
4,71473,82,221,1319.0,557.00,5,5,6.455366,71474,2024-08-11 17:00:00,...,150,{'meta': {'collection-concept-id': 'C256086262...,2024-08-11 16:12:00,2024-08-11 16:18:00,147,{'meta': {'collection-concept-id': 'C255991929...,2024-08-11 15:48:00,2024-08-11 15:54:00,0 days 01:08:37,0 days 01:32:37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,2192481,2487,578,2666.0,676.00,1,5,6.020217,2192482,2024-09-30 19:30:00,...,5975,{'meta': {'collection-concept-id': 'C256086262...,2024-09-30 19:30:00,2024-09-30 19:36:00,5968,{'meta': {'collection-concept-id': 'C255991929...,2024-09-30 19:06:00,2024-09-30 19:12:00,0 days 00:28:29,0 days 00:52:29
975,2193453,2488,580,2666.0,676.00,1,5,6.325706,2193454,2024-09-30 20:00:00,...,5975,{'meta': {'collection-concept-id': 'C256086262...,2024-09-30 19:30:00,2024-09-30 19:36:00,5968,{'meta': {'collection-concept-id': 'C255991929...,2024-09-30 19:06:00,2024-09-30 19:12:00,0 days 00:28:29,0 days 00:52:29
976,2194102,2489,112,744.0,96.00,1,5,5.061385,2194103,2024-09-30 20:30:00,...,5980,{'meta': {'collection-concept-id': 'C256086262...,2024-09-30 20:24:00,2024-09-30 20:30:00,5973,{'meta': {'collection-concept-id': 'C255991929...,2024-09-30 20:00:00,2024-09-30 20:06:00,0 days 00:20:41,0 days 00:44:41
977,2196284,2491,494,2480.0,451.20,5,5,8.857302,2196285,2024-09-30 21:30:00,...,5985,{'meta': {'collection-concept-id': 'C256086262...,2024-09-30 21:18:00,2024-09-30 21:24:00,5978,{'meta': {'collection-concept-id': 'C255991929...,2024-09-30 20:54:00,2024-09-30 21:00:00,0 days 00:01:27,0 days 00:25:27


For each overpass, we need to read the EarthCARE data in the vicinity of the storm, and sample the two sounder overpasses in a given radius

In [6]:
# EarthCARE

In [7]:
from products import PRODUCTS

In [8]:
list(PRODUCTS.keys())

['AC__TC__2B',
 'ACM_CAP_2B',
 'ACM_RT__2B',
 'ALL_DF__2B',
 'ATL_EBD_2A',
 'CPR_CD__2A',
 'CPR_FMR_2A']

#### coloc_gdf.groupby("granule").groups

In [10]:
granule_group = coloc_gdf.groupby("granule").get_group('01150D')

In [11]:
granule_group

,index_coloc,frame,idx,hdim_1,hdim_2,num,threshold_value,max_precip,feature,time,...,index_n20,results_n20,start_time_n20,end_time_n20,index_snpp,results_snpp,start_time_snpp,end_time_snpp,time_diff_n20,time_diff_snpp
1,35675,42,155,750.0,662.0,1,5,5.148579,35676,2024-08-10 21:00:00,...,52,{'meta': {'collection-concept-id': 'C256086262...,2024-08-10 19:54:00,2024-08-10 20:00:00,49,{'meta': {'collection-concept-id': 'C255991929...,2024-08-10 19:30:00,2024-08-10 19:36:00,0 days 01:11:00,0 days 01:35:00
2,36208,42,945,745.0,638.0,1,10,11.937240,36209,2024-08-10 21:00:00,...,52,{'meta': {'collection-concept-id': 'C256086262...,2024-08-10 19:54:00,2024-08-10 20:00:00,49,{'meta': {'collection-concept-id': 'C255991929...,2024-08-10 19:30:00,2024-08-10 19:36:00,0 days 01:11:00,0 days 01:35:00


In [12]:
granule_group.iloc[0][[f'enclosure_{p}' for p in PRODUCTS]]

enclosure_AC__TC__2B    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_ACM_CAP_2B    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_ACM_RT__2B    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_ALL_DF__2B    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_ATL_EBD_2A    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_CPR_CD__2A    https://catalog.maap.eo.esa.int/data/earthcare...
enclosure_CPR_FMR_2A    https://catalog.maap.eo.esa.int/data/earthcare...
Name: 1, dtype: object

In [13]:
granule_group["enclosure_ACM_CAP_2B"].iloc[0]

'https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/ACM_CAP_2B/BA/2024/08/10/ECA_EXBA_ACM_CAP_2B_20240810T210500Z_20250905T082513Z_01150D/ECA_EXBA_ACM_CAP_2B_20240810T210500Z_20250905T082513Z_01150D/ECA_EXBA_ACM_CAP_2B_20240810T210500Z_20250905T082513Z_01150D.h5'

In [14]:
from data_io import read_ec_file

In [15]:
from data_processing import regrid_height, select_vars

In [16]:
from collections import defaultdict

In [17]:
def unfold_doppler(w, w_thresh=5):
    d_w = w.diff("CPR_height")
    folds = np.cumsum(
        ((d_w>w_thresh).astype(int) - (d_w<-w_thresh).astype(int))[:,::-1], 
        1
    )[:,::-1]
    folds = folds - ((folds==folds[:,0])*folds[:,0])

    w_unfold = w.copy()
    w_unfold.data[:,:-1] = w_unfold.data[:,:-1] + folds * 10

    return w_unfold

In [18]:
import earthaccess
auth = earthaccess.login(persist=True)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
def get_sounder_coloc(sounder_dict, lon, lat, dist=1):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        with xr.open_dataset(earthaccess.open(
            [earthaccess.results.DataGranule(sounder_dict)], 
            show_progress=False
        )[0]) as ds:
            ds = ds.where(
                (
                    (ds.lon > (lon - dist))
                    & (ds.lon < (lon + dist))
                    & (ds.lat > (lat - dist))
                    & (ds.lat < (lat + dist))
                ),
                drop=True
            ).load()

    return ds


In [34]:
def load_regrid_ec_granules(
    granule_group, products, heights=np.arange(50, 2e4, 100)[::-1]
):
    datasets = pd.DataFrame(
        index = granule_group.index
    )

    idx_to_remove = []
    
    for encloure, (product, variables) in zip(
        granule_group.iloc[0][[f'enclosure_{p}' for p in PRODUCTS]],
        products.items()
    ):
        with read_ec_file(encloure) as ds:
            ds = select_vars(
                ds, 
                variables, 
                coords=[
                    "time", "latitude", "longitude", "latitude_active", "longitude_active", "height", "height_level", "height_layer"
                ], 
                rename=dict(
                    latitude_active="latitude", longitude_active="longitude"
                )
            ).load()

            ds_geom = gpd.GeoDataFrame(
                geometry=list(shapely.MultiPoint(
                    list(zip(ds.longitude, ds.latitude))
                ).geoms),
                crs=4326, 
            )

            if product == "CPR_CD__2A" and "doppler_velocity_best_estimate" in variables:
                ds["doppler_velocity_unfolded"] = unfold_doppler(ds["doppler_velocity_best_estimate"])

            datasets[product] = [
                regrid_height(
                    ds.isel(
                        along_track=ds_geom.sjoin(granule_group[["geometry"]].iloc[:i+1]).index
                    ),
                    heights
                )
                for i in range(len(granule_group))
            ]

            # product_out = []
            # for i, idx in enumerate(granule_group.index):
            #     try:
            #         datasets.loc[idx, product] = regrid_height(
            #             ds.isel(
            #                 along_track=ds_geom.sjoin(granule_group[["geometry"]].iloc[:i+1]).index
            #             ),
            #             heights
            #         )
            #     except ValueError:
            #         datasets.loc[idx, product] = None
            #         idx_to_remove.append(idx)

        # granule_group = granule_group.loc[~granule_group.index.isin(idx_to_remove)]

    datasets["noaa_20"] = [
        get_sounder_coloc(
            granule_group.results_n20[i], granule_group.lon[i], granule_group.lat[i]
        )
        for i in granule_group.index
    ]

    datasets["snpp"] = [
        get_sounder_coloc(
            granule_group.results_snpp[i], granule_group.lon[i], granule_group.lat[i]
        )
        for i in granule_group.index
    ]
            
    return datasets

In [37]:
granule_group = coloc_gdf.groupby("granule").get_group('01196A')

In [38]:
datasets = load_regrid_ec_granules(granule_group, PRODUCTS)

In [39]:
datasets

,AC__TC__2B,ACM_CAP_2B,ACM_RT__2B,ALL_DF__2B,ATL_EBD_2A,CPR_CD__2A,CPR_FMR_2A,noaa_20,snpp
53,"[synergetic_target_classification, CPR_ATLID_s...","[tropopause_height, synergy_status, quality_st...","[solar_zenith_angle, flux_down_solar_1d_all_sk...","[BBR_effective_flux_solar, BBR_effective_flux_...","[simple_classification, quality_status, partic...","[quality_status, doppler_velocity_and_spectrum...","[surface_elevation, land_flag, quality_status,...","[obs_id, obs_time_tai93, obs_time_utc, lat_geo...","[obs_id, obs_time_tai93, obs_time_utc, lat_geo..."
55,"[synergetic_target_classification, CPR_ATLID_s...","[tropopause_height, synergy_status, quality_st...","[solar_zenith_angle, flux_down_solar_1d_all_sk...","[BBR_effective_flux_solar, BBR_effective_flux_...","[simple_classification, quality_status, partic...","[quality_status, doppler_velocity_and_spectrum...","[surface_elevation, land_flag, quality_status,...","[obs_id, obs_time_tai93, obs_time_utc, lat_geo...","[obs_id, obs_time_tai93, obs_time_utc, lat_geo..."


In [43]:
for idx, r in datasets.iterrows():
    break

In [286]:
xr.DataTree(
    children={
        k:xr.DataTree(v) for k,v in r.to_dict().items()
    }
).to_netcdf(f'overpass_{idx:05d}.nc')

In [44]:
save_path = Path.home() / "my-public-bucket" / "orcestra" / "overpass"
save_path.mkdir(exist_ok=True, parents=True)

In [45]:
save_path.exists()

True

In [48]:
for granule, group in coloc_gdf.loc[979:].groupby("granule"):
    print(granule)
    try:
        datasets = load_regrid_ec_granules(group, PRODUCTS)
    except ValueError:
        pass
    else:
        for idx, r in datasets.iterrows():
            print(idx)
            xr.DataTree(
                children={
                    k:xr.DataTree(v) for k,v in r.to_dict().items()
                }
            ).to_netcdf(save_path / f'overpass_{idx:05d}_{granule}.nc')